# XML Demo

NOTE: Tracemalloc is used to track performance and present it nicely

In [1]:
import lxml.etree as ET
import tracemalloc
import os

## 1. Create sample XML file

In [ ]:
# You can ignore this cell - it's just for creating a sample XML file to work with

def create_sample_xml(filename, num_records=100_000):
    print(f"Creating {filename} with {num_records} <record> elements...")
    with open(filename, "wb") as f:
        f.write(b'<?xml version="1.0"?>\n<root>\n')
        for i in range(num_records):
            record = f'  <record id="{i}">\n    <text>Sample sentence number {i}</text>\n  </record>\n'
            f.write(record.encode())
        f.write(b'</root>')
    file_size = os.path.getsize(filename) / (1024 * 1024)
    print(f"File created. Size: {file_size:.2f} MB\n")

## 2. Memory efficient parsing and cleanup

In [ ]:
def process_xml_iterparse(filename):
    tracemalloc.start()
    
    count = 0
    # iterparse yields (event, element) for each 'record' tag
    for event, elem in ET.iterparse(filename, tag='record', events=('end',)):
        # Extract data (simulate processing)
        text = elem.findtext('text')
        # Free the element
        elem.clear()
        # Remove previous siblings to prevent memory leak
        while elem.getprevious() is not None:
            parent = elem.getparent()
            if parent is not None:
                parent.remove(elem.getprevious())
        count += 1
        if count >= 100_000:  # early stop for demo
            break
    
    current, peak = tracemalloc.get_traced_memory()
    peak_mb = peak / (1024 * 1024)
    tracemalloc.stop()
    print(f"[iterparse + clear] Processed {count} records")
    print(f"[iterparse + clear] Peak memory usage: {peak_mb:.2f} MB")
    return peak_mb

## 3. Memory inefficient processing

In [ ]:
def process_xml_full_parse(filename):
    tracemalloc.start()
    
    # Parses the whole file into an ElementTree object
    tree = ET.parse(filename)
    root = tree.getroot()
    
    count = 0
    for elem in root.findall('record'):
        text = elem.findtext('text')
        count += 1
        if count >= 100_000:
            break
    
    current, peak = tracemalloc.get_traced_memory()
    peak_mb = peak / (1024 * 1024)
    tracemalloc.stop()
    print(f"[ET.parse] Processed {count} records")
    print(f"[ET.parse] Peak memory usage: {peak_mb:.2f} MB")
    return peak_mb

## Comparison

In [ ]:
filename = "large_data.xml"
create_sample_xml(filename, num_records=200_000)  # 200k records

print("=" * 50)
print("Running iterparse with cleanup...")
mem_iter = process_xml_iterparse(filename)

print("\n" + "=" * 50)
print("Running full tree parse...")
mem_full = process_xml_full_parse(filename)

print("\n" + "=" * 50)
print(f"Memory saved with iterparse+clear: {mem_full - mem_iter:.2f} MB")